# Constrained Behavioural Evidence Layer

## Overview

This notebook applies individual-level travel-mode feasibility constraints to the unconstrained mobility behaviour profile constructed from the Representative Demographic Profile Space and the integrated National Travel Survey evidence.

The original behavioural profile contains three conditional probability distributions:

- trip-purpose probabilities conditioned on age and gender;
- start-time probabilities conditioned on trip purpose; and
- travel-mode probabilities conditioned on trip purpose.

The trip-purpose distribution is retained without modification because age and gender are already included in the conditioning structure of NTS0611. Similarly, the start-time distributions are retained because no individual-level time constraints are currently applied.

However, the NTS0409 travel-mode distributions are conditioned only on trip purpose and do not account for the demographic characteristics of an individual profile. As a result, they may retain travel modes that are not feasible for a particular age group. For example, profiles aged 0–4 or 5–11 cannot travel as a **Car or van driver**.

This notebook therefore:

1. identifies travel modes that are infeasible for the selected demographic profile;
2. removes the infeasible modes from each trip-purpose branch;
3. renormalises each affected travel-mode distribution so that it sums to one; and
4. validates that the trip-purpose and start-time distributions remain unchanged.

The output is a **Demographically Constrained Mobility Behaviour Profile** that retains the factorised probability structure:

\[
P(\text{Purpose}\mid\text{Age, Gender})
\]

\[
P(\text{Start Time}\mid\text{Purpose})
\]

\[
P^*(\text{Travel Mode}\mid\text{Purpose},\text{Demographics})
\]

This notebook does not sample a journey, generate a synthetic population, or simulate travel.

## 1. Import Libraries

In [1]:
import json
from copy import deepcopy
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

## 2. Set Up File Paths

In [2]:
project_root = Path.cwd().parents[1]

population_file = (
    project_root
    / "output"
    / "demographic_profiles"
    / "representative_population.parquet"
)

behavioural_evidence_file = (
    project_root
    / "output"
    / "behavioural_evidence"
    / "nts_behavioural_evidence.json"
)

## 3. Load Input Data

In [3]:
population = pd.read_parquet(population_file)

with open(
    behavioural_evidence_file,
    "r",
    encoding="utf-8",
) as file:
    behavioural_evidence = json.load(file)

## 4. Inspect Input Data

In [4]:
print(f"Population shape: {population.shape}")
print(f"Population columns: {population.columns.tolist()}")

print(
    "\nBehavioural evidence type:",
    type(behavioural_evidence).__name__,
)

print(
    "Number of trip purposes:",
    len(behavioural_evidence),
)

print(
    "Available trip purposes:",
    list(behavioural_evidence.keys()),
)

Population shape: (2869120, 14)
Population columns: ['Age', 'Gender', 'Household Type', 'Household Count', 'Economic Activity Status', 'Economic Count', 'Industry', 'Industry Count', 'Ethnic Group', 'Ethnic Count', 'Distance travelled to work', 'Travel Distance Count', 'Car Ownership', 'Car Ownership Count']

Behavioural evidence type: dict
Number of trip purposes: 14
Available trip purposes: ['Commuting', 'Business', 'Education', 'Escort Education', 'Shopping', 'Other Escort', 'Personal Business', 'Visiting Friends at Private Home', 'Visiting Friends Elsewhere', 'Entertainment or Public Activity', 'Sport: Participate', 'Holiday: Base', 'Day Trip', 'Just Walk']


In [5]:
example_purpose = "Commuting"

print(
    f"Evidence components for {example_purpose}:"
)

print(
    behavioural_evidence[
        example_purpose
    ].keys()
)

Evidence components for Commuting:
dict_keys(['baseline_profile', 'travel_mode_profile', 'temporal_pattern_profile'])


## 5. Select a Test Representative Profile

A feasible representative profile aged 0–4 is selected from the demographic profile space to test the constraint process.

This profile is used only as a test case. Selecting the first matching row does not represent population sampling and does not imply that the selected row is more representative than other feasible profiles.

The 0–4 age group is selected because it provides clear test conditions:

- **Commuting** is not a feasible trip purpose;
- **Business** is not a feasible trip purpose; and
- **Car or van driver** is not a feasible travel mode.

In [6]:
test_candidates = population[
    (population["Age"] == "0-4")
    & (population["Gender"] == "Female")
]

if test_candidates.empty:
    raise ValueError(
        "No feasible profile found for "
        "Age=0-4 and Gender=Female."
    )

representative_profile = test_candidates.iloc[0]

In [7]:
profile_attribute_columns = [
    "Age",
    "Gender",
    "Household Type",
    "Economic Activity Status",
    "Industry",
    "Ethnic Group",
    "Distance travelled to work",
    "Car Ownership",
]

profile_attributes = representative_profile[
    profile_attribute_columns
].to_dict()

In [8]:
print("Selected test profile:")

for attribute, value in profile_attributes.items():
    print(f"- {attribute}: {value}")

Selected test profile:
- Age: 0-4
- Gender: Female
- Household Type: Couple household: With dependent children
- Economic Activity Status: Does not apply
- Industry: Does not apply
- Ethnic Group: Asian, Asian British or Asian Welsh: Bangladeshi
- Distance travelled to work: Does not apply
- Car Ownership: 1 car or van in household


## 6. Construct the Unconstrained Behaviour Profile

Before applying demographic constraints, an unconstrained behaviour profile is constructed for the selected test profile.

The profile combines:

- the trip-purpose distribution associated with the profile's age and gender;
- the 24-hour start-time distribution associated with each trip purpose; and
- the 13-mode travel distribution associated with each trip purpose.

This unconstrained profile provides the baseline against which the effects of the demographic constraints can be evaluated.

### Dynamic Evidence Validation

Temporal and travel-mode categories are identified directly from the input data. Descriptive columns (`Year`, `Trip purpose`, and `Purpose`) are excluded because they are not probabilities. No fixed category counts are used, allowing the function to remain valid if the input datasets change.

In [9]:
def construct_unconstrained_behaviour_profile(
    profile,
    behavioural_evidence,
):
    """Construct an unconstrained mobility behaviour profile."""

    required_attributes = {
        "Age",
        "Gender",
        "Household Type",
        "Economic Activity Status",
        "Industry",
        "Ethnic Group",
        "Distance travelled to work",
        "Car Ownership",
    }

    missing = required_attributes - set(profile)

    if missing:
        raise ValueError(
            f"Missing profile attributes: {sorted(missing)}"
        )

    age = profile["Age"]
    gender = profile["Gender"]

    temporal_non_probability_columns = {
        "Year",
        "Trip purpose",
    }

    mode_non_probability_columns = {
        "Purpose",
    }

    first_evidence = next(
        iter(behavioural_evidence.values())
    )

    expected_time_columns = (
        set(first_evidence["temporal_pattern_profile"][0])
        - temporal_non_probability_columns
    )

    expected_mode_columns = (
        set(first_evidence["travel_mode_profile"][0])
        - mode_non_probability_columns
    )

    mobility_behaviour = {}

    for purpose, evidence in behavioural_evidence.items():

        purpose_matches = [
            record
            for record in evidence["baseline_profile"]
            if record["Age"] == age
            and record["Gender"] == gender
        ]

        if len(purpose_matches) != 1:
            raise ValueError(
                f"Expected one purpose probability for "
                f"{purpose}; found {len(purpose_matches)}."
            )

        purpose_probability = purpose_matches[0][
            "Trip Purpose Probability"
        ]

        temporal_records = evidence[
            "temporal_pattern_profile"
        ]

        mode_records = evidence["travel_mode_profile"]

        if len(temporal_records) != 1:
            raise ValueError(
                f"Expected one temporal record for {purpose}."
            )

        if len(mode_records) != 1:
            raise ValueError(
                f"Expected one travel-mode record for {purpose}."
            )

        temporal_record = temporal_records[0]
        mode_record = mode_records[0]

        time_probabilities = {
            column: probability
            for column, probability in temporal_record.items()
            if column not in temporal_non_probability_columns
        }

        mode_probabilities = {
            column: probability
            for column, probability in mode_record.items()
            if column not in mode_non_probability_columns
        }

        if set(time_probabilities) != expected_time_columns:
            raise ValueError(
                f"Inconsistent temporal columns for {purpose}."
            )

        if set(mode_probabilities) != expected_mode_columns:
            raise ValueError(
                f"Inconsistent travel-mode columns for {purpose}."
            )

        if (
            any(
                pd.isna(p) or not 0 <= p <= 1
                for p in time_probabilities.values()
            )
            or abs(sum(time_probabilities.values()) - 1) >= 1e-8
        ):
            raise ValueError(
                f"Invalid temporal distribution for {purpose}."
            )

        if (
            any(
                pd.isna(p) or not 0 <= p <= 1
                for p in mode_probabilities.values()
            )
            or abs(sum(mode_probabilities.values()) - 1) >= 1e-8
        ):
            raise ValueError(
                f"Invalid travel-mode distribution for {purpose}."
            )

        mobility_behaviour[purpose] = {
            "trip_purpose_probability": purpose_probability,
            "temporal_distribution": {
                "year": temporal_record["Year"],
                "start_time_probabilities": time_probabilities,
            },
            "travel_mode_distribution": {
                "travel_mode_probabilities": mode_probabilities,
            },
        }

    purpose_sum = sum(
        values["trip_purpose_probability"]
        for values in mobility_behaviour.values()
    )

    if abs(purpose_sum - 1) >= 1e-8:
        raise ValueError(
            f"Trip-purpose probabilities do not sum to 1: "
            f"{purpose_sum}."
        )

    return {
        "representative_profile": deepcopy(profile),
        "mobility_behaviour": mobility_behaviour,
    }

In [10]:
unconstrained_behaviour_profile = (
    construct_unconstrained_behaviour_profile(
        profile=profile_attributes,
        behavioural_evidence=behavioural_evidence,
    )
)

In [11]:
unconstrained_summary = pd.DataFrame(
    [
        {
            "Trip Purpose": purpose,
            "Purpose Probability": values[
                "trip_purpose_probability"
            ],
            "Time Periods": len(
                values["temporal_distribution"][
                    "start_time_probabilities"
                ]
            ),
            "Travel Modes": len(
                values["travel_mode_distribution"][
                    "travel_mode_probabilities"
                ]
            ),
        }
        for purpose, values
        in unconstrained_behaviour_profile[
            "mobility_behaviour"
        ].items()
    ]
).sort_values(
    "Purpose Probability",
    ascending=False,
    ignore_index=True,
)

display(unconstrained_summary)

,Trip Purpose,Purpose Probability,Time Periods,Travel Modes
0,Education,0.327535,24,11
1,Other Escort,0.142315,24,11
2,Visiting Friends at Private Home,0.106521,24,11
3,Entertainment or Public Activity,0.078185,24,11
4,Personal Business,0.075573,24,11
5,Just Walk,0.053662,24,11
6,Escort Education,0.044123,24,11
7,Visiting Friends Elsewhere,0.043625,24,11
8,Day Trip,0.041132,24,11
9,Sport: Participate,0.035980,24,11


## 7. Define Travel-Mode Feasibility Constraints

The integrated NTS0409 travel-mode distributions are conditioned on trip purpose but not on individual demographic attributes.

This section therefore defines a small set of deterministic constraints for travel modes that are clearly infeasible for particular demographic groups.

In the initial implementation, **Car or van driver** is excluded for profiles aged 0–4 and 5–11. No deterministic driver exclusion is applied to the 12–17 age group because this category includes 17-year-olds who may legally learn to drive.

The constraints are stored separately from the probability-processing function so that additional evidence-supported rules can be added later.

In [12]:
example_purpose = "Commuting"

example_mode_record = behavioural_evidence[
    example_purpose
]["travel_mode_profile"][0]

available_travel_modes = {
    mode
    for mode in example_mode_record
    if mode != "Purpose"
}

print("Available travel modes:")

for mode in sorted(available_travel_modes):
    print(f"- {mode}")

Available travel modes:
- Car or van driver
- Car or van passenger
- Motorcycle
- Non-local bus
- Other Private Transport
- Other Public Transport
- Other local bus
- Pedal Cycle
- Surface Rail
- Taxi or minicab
- Walk


In [13]:
travel_mode_constraints_by_age = {
    "0-4": {
        "Car or van driver",
    },
    "5-11": {
        "Car or van driver",
    },
}

In [14]:
constrained_travel_modes = set().union(
    *travel_mode_constraints_by_age.values()
)

unknown_travel_modes = (
    constrained_travel_modes
    - available_travel_modes
)

if unknown_travel_modes:
    raise ValueError(
        "Unknown constrained travel modes: "
        f"{sorted(unknown_travel_modes)}"
    )

print("Travel-mode constraint-rule validation: PASS")

Travel-mode constraint-rule validation: PASS


In [15]:
test_age = profile_attributes["Age"]

excluded_travel_modes = (
    travel_mode_constraints_by_age.get(
        test_age,
        set(),
    )
)

print(f"Test profile age: {test_age}")
print(
    "Excluded travel modes:",
    sorted(excluded_travel_modes),
)

Test profile age: 0-4
Excluded travel modes: ['Car or van driver']


## 8. Apply and Renormalise Travel-Mode Constraints

The travel-mode feasibility constraints are applied separately within each trip-purpose branch.

For the selected profile aged 0–4, **Car or van driver** is removed from every purpose-conditioned travel-mode distribution. The remaining mode probabilities are then renormalised within each trip purpose:

\[
P^*(m \mid p,d)
=
\frac{P(m \mid p)}
{\sum_{m' \in F(p,d)} P(m' \mid p)}
\]

where \(F(p,d)\) represents the travel modes that remain feasible for trip purpose \(p\) and demographic profile \(d\).

The trip-purpose probabilities and start-time distributions are not modified during this process.

### Apply Age-Based Travel-Mode Constraints

This function removes `Car or van driver` from the travel-mode distributions of profiles aged 0–4 or 5–11. The remaining probabilities are renormalised within each trip-purpose branch. All other age groups retain their original travel-mode distributions.

In [16]:
def apply_travel_mode_constraints(
    behaviour_profile,
    constraints_by_age,
):
    """Remove infeasible modes and renormalise each mode distribution."""

    constrained_profile = deepcopy(
        behaviour_profile
    )

    profile = constrained_profile[
        "representative_profile"
    ]

    mobility_behaviour = constrained_profile[
        "mobility_behaviour"
    ]

    age = profile["Age"]

    excluded_modes = constraints_by_age.get(
        age,
        set(),
    )

    mode_adjustments = {}

    for purpose, values in mobility_behaviour.items():

        mode_probabilities = values[
            "travel_mode_distribution"
        ]["travel_mode_probabilities"]

        applicable_exclusions = (
            excluded_modes
            & set(mode_probabilities)
        )

        removed_probability_mass = sum(
            mode_probabilities[mode]
            for mode in applicable_exclusions
        )

        for mode in applicable_exclusions:
            del mode_probabilities[mode]

        if not mode_probabilities:
            raise ValueError(
                "All travel modes were removed for "
                f"purpose={purpose}, age={age}."
            )

        remaining_probability_mass = sum(
            mode_probabilities.values()
        )

        if remaining_probability_mass <= 0:
            raise ValueError(
                "The remaining travel-mode probability "
                "mass must be greater than zero for "
                f"purpose={purpose}."
            )

        for mode in mode_probabilities:
            mode_probabilities[mode] = (
                mode_probabilities[mode]
                / remaining_probability_mass
            )

        renormalised_probability_sum = sum(
            mode_probabilities.values()
        )

        if abs(
            renormalised_probability_sum - 1.0
        ) >= 1e-8:
            raise ValueError(
                "Renormalised travel-mode probabilities "
                "do not sum to 1 for "
                f"purpose={purpose}: "
                f"{renormalised_probability_sum}."
            )

        mode_adjustments[purpose] = {
            "excluded_modes": sorted(
                applicable_exclusions
            ),
            "removed_probability_mass": (
                removed_probability_mass
            ),
            "remaining_probability_mass_before_renormalisation": (
                remaining_probability_mass
            ),
            "renormalisation_factor": (
                1.0 / remaining_probability_mass
            ),
        }

    constrained_profile[
        "constraint_metadata"
    ] = {
        "profile_age": age,
        "travel_mode_adjustments": (
            mode_adjustments
        ),
    }

    return constrained_profile

In [17]:
constrained_behaviour_profile = (
    apply_travel_mode_constraints(
        behaviour_profile=(
            unconstrained_behaviour_profile
        ),
        constraints_by_age=(
            travel_mode_constraints_by_age
        ),
    )
)

In [18]:
mode_constraint_summary = pd.DataFrame(
    [
        {
            "Trip Purpose": purpose,
            "Driver Probability Before": (
                unconstrained_behaviour_profile[
                    "mobility_behaviour"
                ][purpose][
                    "travel_mode_distribution"
                ][
                    "travel_mode_probabilities"
                ].get(
                    "Car or van driver",
                    0.0,
                )
            ),
            "Driver Probability After": (
                constrained_behaviour_profile[
                    "mobility_behaviour"
                ][purpose][
                    "travel_mode_distribution"
                ][
                    "travel_mode_probabilities"
                ].get(
                    "Car or van driver",
                    0.0,
                )
            ),
            "Modes Before": len(
                unconstrained_behaviour_profile[
                    "mobility_behaviour"
                ][purpose][
                    "travel_mode_distribution"
                ][
                    "travel_mode_probabilities"
                ]
            ),
            "Modes After": len(
                constrained_behaviour_profile[
                    "mobility_behaviour"
                ][purpose][
                    "travel_mode_distribution"
                ][
                    "travel_mode_probabilities"
                ]
            ),
            "Probability Sum After": sum(
                constrained_behaviour_profile[
                    "mobility_behaviour"
                ][purpose][
                    "travel_mode_distribution"
                ][
                    "travel_mode_probabilities"
                ].values()
            ),
        }
        for purpose
        in unconstrained_behaviour_profile[
            "mobility_behaviour"
        ]
    ]
)

display(mode_constraint_summary)

,Trip Purpose,Driver Probability Before,Driver Probability After,Modes Before,Modes After,Probability Sum After
0,Commuting,0.543133,0.0,11,10,1.0
1,Business,0.686180,0.0,11,10,1.0
2,Education,0.234031,0.0,11,10,1.0
3,Escort Education,0.234031,0.0,11,10,1.0
4,Shopping,0.457349,0.0,11,10,1.0
5,Other Escort,0.564046,0.0,11,10,1.0
6,Personal Business,0.425456,0.0,11,10,1.0
7,Visiting Friends at Private Home,0.379552,0.0,11,10,1.0
8,Visiting Friends Elsewhere,0.379552,0.0,11,10,1.0
9,Entertainment or Public Activity,0.379552,0.0,11,10,1.0


## 9. Validate Travel-Mode Constraints Across Age Groups

The constraint function is tested using representative profiles from different age groups.

The expected behaviour is:

- **0–4:** Car or van driver is excluded;
- **5–11:** Car or van driver is excluded;
- **12–17:** no deterministic exclusion is applied because this age group includes 17-year-olds who may legally learn to drive; and
- **23–34:** no age-based driver exclusion is applied.

These test cases verify that the constraint function responds to the demographic profile rather than removing the driver mode from every profile.

In [19]:
def construct_age_test_case(
    population,
    age,
    gender,
    behavioural_evidence,
):
    """Construct unconstrained and constrained profiles for testing."""

    candidates = population[
        (population["Age"] == age)
        & (population["Gender"] == gender)
    ]

    if candidates.empty:
        raise ValueError(
            "No feasible profile found for "
            f"age={age}, gender={gender}."
        )

    selected_row = candidates.iloc[0]

    selected_attributes = selected_row[
        profile_attribute_columns
    ].to_dict()

    unconstrained_profile = (
        construct_unconstrained_behaviour_profile(
            profile=selected_attributes,
            behavioural_evidence=behavioural_evidence,
        )
    )

    constrained_profile = (
        apply_travel_mode_constraints(
            behaviour_profile=unconstrained_profile,
            constraints_by_age=(
                travel_mode_constraints_by_age
            ),
        )
    )

    return {
        "profile_attributes": selected_attributes,
        "unconstrained_profile": (
            unconstrained_profile
        ),
        "constrained_profile": (
            constrained_profile
        ),
    }

In [20]:
age_test_groups = [
    "0-4",
    "5-11",
    "12-17",
    "23-34",
]

age_test_results = {
    age: construct_age_test_case(
        population=population,
        age=age,
        gender="Female",
        behavioural_evidence=(
            behavioural_evidence
        ),
    )
    for age in age_test_groups
}

In [21]:
constrained_age_groups = {
    "0-4",
    "5-11",
}

age_validation_records = []

for age, test_result in age_test_results.items():

    unconstrained = test_result[
        "unconstrained_profile"
    ]["mobility_behaviour"]

    constrained = test_result[
        "constrained_profile"
    ]["mobility_behaviour"]

    driver_exclusion_expected = (
        age in constrained_age_groups
    )

    mode_structure_valid = True
    mode_sums_valid = True
    purpose_unchanged = True
    temporal_unchanged = True
    mode_counts_after = set()

    for purpose, constrained_values in constrained.items():

        original_values = unconstrained[purpose]

        original_modes = original_values[
            "travel_mode_distribution"
        ]["travel_mode_probabilities"]

        constrained_modes = constrained_values[
            "travel_mode_distribution"
        ]["travel_mode_probabilities"]

        expected_modes = set(original_modes)

        if driver_exclusion_expected:
            expected_modes.discard("Car or van driver")

        if set(constrained_modes) != expected_modes:
            mode_structure_valid = False

        if abs(sum(constrained_modes.values()) - 1) >= 1e-8:
            mode_sums_valid = False

        if (
            constrained_values["trip_purpose_probability"]
            != original_values["trip_purpose_probability"]
        ):
            purpose_unchanged = False

        if (
            constrained_values["temporal_distribution"]
            != original_values["temporal_distribution"]
        ):
            temporal_unchanged = False

        mode_counts_after.add(len(constrained_modes))

    assert mode_structure_valid
    assert mode_sums_valid
    assert purpose_unchanged
    assert temporal_unchanged

    age_validation_records.append(
        {
            "Age": age,
            "Driver Exclusion Expected": (
                driver_exclusion_expected
            ),
            "Mode Counts After": sorted(mode_counts_after),
            "Mode Structure Valid": mode_structure_valid,
            "All Mode Sums Valid": mode_sums_valid,
            "Purpose Evidence Unchanged": purpose_unchanged,
            "Temporal Evidence Unchanged": temporal_unchanged,
            "Validation": "PASS",
        }
    )

In [22]:
age_validation_summary = pd.DataFrame(
    age_validation_records
)

display(age_validation_summary)

print(
    "Cross-age travel-mode constraint "
    "validation: PASS"
)

,Age,Driver Exclusion Expected,Mode Counts After,Mode Structure Valid,All Mode Sums Valid,Purpose Evidence Unchanged,Temporal Evidence Unchanged,Validation
0,0-4,True,[10],True,True,True,True,PASS
1,5-11,True,[10],True,True,True,True,PASS
2,12-17,False,[11],True,True,True,True,PASS
3,23-34,False,[11],True,True,True,True,PASS


Cross-age travel-mode constraint validation: PASS


## 10. Inspect the Final Constrained Output

The final output is inspected to confirm that it retains:

- the selected representative demographic profile;
- all trip-purpose branches from the behavioural evidence;
- the complete start-time distribution for each trip purpose;
- only the travel modes that remain feasible for the selected profile; and
- travel-mode distributions that sum to one.

The numbers of trip purposes, temporal categories, and travel modes are reported for inspection only. They are inferred from the input evidence and are not used as fixed validation requirements.

In [23]:
final_behaviour = (
    constrained_behaviour_profile[
        "mobility_behaviour"
    ]
)

final_output_summary = pd.DataFrame(
    [
        {
            "Trip Purpose": purpose,
            "Purpose Probability": values[
                "trip_purpose_probability"
            ],
            "Time Periods": len(
                values["temporal_distribution"][
                    "start_time_probabilities"
                ]
            ),
            "Travel Modes": len(
                values["travel_mode_distribution"][
                    "travel_mode_probabilities"
                ]
            ),
            "Mode Probability Sum": sum(
                values["travel_mode_distribution"][
                    "travel_mode_probabilities"
                ].values()
            ),
            "Driver Present": (
                "Car or van driver"
                in values[
                    "travel_mode_distribution"
                ][
                    "travel_mode_probabilities"
                ]
            ),
        }
        for purpose, values
        in final_behaviour.items()
    ]
).sort_values(
    "Purpose Probability",
    ascending=False,
    ignore_index=True,
)

display(final_output_summary)

,Trip Purpose,Purpose Probability,Time Periods,Travel Modes,Mode Probability Sum,Driver Present
0,Education,0.327535,24,10,1.0,False
1,Other Escort,0.142315,24,10,1.0,False
2,Visiting Friends at Private Home,0.106521,24,10,1.0,False
3,Entertainment or Public Activity,0.078185,24,10,1.0,False
4,Personal Business,0.075573,24,10,1.0,False
5,Just Walk,0.053662,24,10,1.0,False
6,Escort Education,0.044123,24,10,1.0,False
7,Visiting Friends Elsewhere,0.043625,24,10,1.0,False
8,Day Trip,0.041132,24,10,1.0,False
9,Sport: Participate,0.035980,24,10,1.0,False


In [24]:
print(
    "Final output keys:",
    constrained_behaviour_profile.keys(),
)

print(
    "Representative profile attributes:",
    len(
        constrained_behaviour_profile[
            "representative_profile"
        ]
    ),
)

print(
    "Trip-purpose branches:",
    len(final_behaviour),
)

print(
    "Constraint metadata available:",
    "constraint_metadata"
    in constrained_behaviour_profile,
)

Final output keys: dict_keys(['representative_profile', 'mobility_behaviour', 'constraint_metadata'])
Representative profile attributes: 8
Trip-purpose branches: 14
Constraint metadata available: True


In [25]:
inspection_purpose = "Education"

inspection_values = final_behaviour[
    inspection_purpose
]

trip_purpose_probability = inspection_values[
    "trip_purpose_probability"
]

temporal_year = inspection_values[
    "temporal_distribution"
]["year"]

print(f"Trip Purpose: {inspection_purpose}")
print(
    "Trip Purpose Probability: "
    f"{trip_purpose_probability:.8f}"
)
print(f"Temporal Evidence Year: {temporal_year}")

Trip Purpose: Education
Trip Purpose Probability: 0.32753540
Temporal Evidence Year: 2023 to 2024


In [26]:
representative_profile_df = pd.DataFrame(
    constrained_behaviour_profile[
        "representative_profile"
    ].items(),
    columns=[
        "Demographic Attribute",
        "Value",
    ],
)

print("\nRepresentative Profile")
display(representative_profile_df)


Representative Profile


,Demographic Attribute,Value
0,Age,0-4
1,Gender,Female
2,Household Type,Couple household: With dependent children
3,Economic Activity Status,Does not apply
4,Industry,Does not apply
5,Ethnic Group,"Asian, Asian British or Asian Welsh: Bangladeshi"
6,Distance travelled to work,Does not apply
7,Car Ownership,1 car or van in household


In [27]:
temporal_probabilities = inspection_values[
    "temporal_distribution"
]["start_time_probabilities"]

temporal_distribution_df = pd.DataFrame(
    temporal_probabilities.items(),
    columns=[
        "Time Period",
        "Probability",
    ],
)

temporal_distribution_df[
    "Percentage"
] = (
    temporal_distribution_df[
        "Probability"
    ]
    * 100
)

print(
    f"\nTemporal Distribution: "
    f"{inspection_purpose}"
)

display(
    temporal_distribution_df.style.format(
        {
            "Probability": "{:.8f}",
            "Percentage": "{:.4f}%",
        }
    )
)


Temporal Distribution: Education


,Time Period,Probability,Percentage
0,0000 to 0059 (%),0.00000000,0.0000%
1,0100 to 0159 (%),0.00000000,0.0000%
2,0200 to 0259 (%),0.00000000,0.0000%
3,0300 to 0359 (%),0.00030642,0.0306%
4,0400 to 0459 (%),0.00004397,0.0044%
5,0500 to 0559 (%),0.00011165,0.0112%
6,0600 to 0659 (%),0.00253959,0.2540%
7,0700 to 0759 (%),0.09799807,9.7998%
8,0800 to 0859 (%),0.37755919,37.7559%
9,0900 to 0959 (%),0.01758576,1.7586%


In [28]:
travel_mode_probabilities = inspection_values[
    "travel_mode_distribution"
]["travel_mode_probabilities"]

travel_mode_distribution_df = pd.DataFrame(
    travel_mode_probabilities.items(),
    columns=[
        "Travel Mode",
        "Probability",
    ],
)

travel_mode_distribution_df[
    "Percentage"
] = (
    travel_mode_distribution_df[
        "Probability"
    ]
    * 100
)

travel_mode_distribution_df = (
    travel_mode_distribution_df
    .sort_values(
        "Probability",
        ascending=False,
        ignore_index=True,
    )
)

print(
    f"\nConstrained Travel-Mode Distribution: "
    f"{inspection_purpose}"
)

display(
    travel_mode_distribution_df.style.format(
        {
            "Probability": "{:.8f}",
            "Percentage": "{:.4f}%",
        }
    )
)


Constrained Travel-Mode Distribution: Education


,Travel Mode,Probability,Percentage
0,Walk,0.52828838,52.8288%
1,Car or van passenger,0.30240574,30.2406%
2,Other local bus,0.08770777,8.7708%
3,Other Private Transport,0.03357845,3.3578%
4,Pedal Cycle,0.01529928,1.5299%
5,Surface Rail,0.01513883,1.5139%
6,Other Public Transport,0.00928639,0.9286%
7,Taxi or minicab,0.00806448,0.8064%
8,Motorcycle,0.00023067,0.0231%
9,Non-local bus,0.00000000,0.0000%


In [29]:
print("Constrained behaviour profile completed.")
print(
    "Trip-purpose branches:",
    len(
        constrained_behaviour_profile[
            "mobility_behaviour"
        ]
    ),
)

Constrained behaviour profile completed.
Trip-purpose branches: 14
